# 04 — Modeling

Train and evaluate 8+ classification algorithms with stratified 5-fold CV.

**Requirements:** FR-10 through FR-14 (PRD)

In [ ]:
import sys
from pathlib import Path

import joblib
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.feature_engineering import engineer_features, get_feature_columns
from src.modeling import compare_all_models, evaluate_on_holdout, get_models, train_test_split_data
from src.utils import data_path, results_path, set_seed

set_seed()
TABLE_DIR = results_path('tables')
MODEL_DIR = results_path('models')
TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load feature-engineered data
df = pd.read_csv(data_path('processed', 'oasis_merged_final.csv'))
if 'BrainAtrophyRatio' not in df.columns:
    df = engineer_features(df)

feature_cols = [c for c in get_feature_columns() if c in df.columns]
X = df[feature_cols]
y = df['target']
print(X.shape, y.value_counts().to_dict())

In [ ]:
# FR-10 to FR-14: Compare all models with 5-fold CV + SMOTE
results_df = compare_all_models(X, y, use_smote=True)
results_df

In [ ]:
# Save results table
results_df.to_csv(TABLE_DIR / 'model_comparison.csv', index=False)
print(f'Saved to {TABLE_DIR / "model_comparison.csv"}')

In [ ]:
# Train/test split and evaluate best model on holdout set
X_train, X_test, y_train, y_test = train_test_split_data(X, y)

best_model_name = 'XGBoost'  # Update after reviewing CV results
best_model = get_models()[best_model_name]
holdout_metrics = evaluate_on_holdout(best_model, X_train, X_test, y_train, y_test)
holdout_metrics

In [ ]:
# Save best model
from src.modeling import build_smote_pipeline

pipeline = build_smote_pipeline(best_model)
pipeline.fit(X_train, y_train)
joblib.dump(pipeline, MODEL_DIR / 'best_model.pkl')
print(f'Saved best model to {MODEL_DIR / "best_model.pkl"}')